# Action-Aware NanoWM — Day 1 Colab bootstrap

This notebook validates the Colab runtime and collects exactly 10 deterministic VizDoom Basic pilot episodes. It does not download the VAE, run a model forward pass, or start training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

REPO = Path('/content/NanoWMActionAware')
if not REPO.exists():
    !git clone https://github.com/ChiCoTheLaAnh/NanoWMActionAware.git {REPO}
%cd {REPO}
PILOT_DIR = Path('/content/drive/MyDrive/ActionAwareNanoWM/data/vizdoom_basic/pilot')
PILOT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['VIZDOOM_DATA_DIR'] = str(PILOT_DIR.parent)
print('Pilot output:', PILOT_DIR)

In [ ]:
# Keep Colab's CUDA-compatible torch and torchvision; install only project dependencies.
%pip install -q -r requirements-colab.txt

In [ ]:
import datetime as dt
import importlib.metadata as metadata
import json
import platform
import subprocess
import sys

import diffusers
import hydra
import torch
import vizdoom
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

expected_versions = {
    'decord': '0.6.0',
    'diffusers': '0.24.0',
    'hydra-core': '1.3.2',
    'pytorch-fid': '0.3.0',
    'vizdoom': '1.3.0',
}
installed_versions = {name: metadata.version(name) for name in expected_versions}
version_mismatches = {
    name: {'expected': expected_versions[name], 'installed': installed}
    for name, installed in installed_versions.items()
    if installed != expected_versions[name]
}
assert not version_mismatches, f'Pinned dependency mismatch: {version_mismatches}'
assert diffusers.__version__ == installed_versions['diffusers'], (
    'The kernel has a stale diffusers import. Factory-reset the runtime and Run all.'
)

sys.path.insert(0, str(REPO / 'src'))
from wm_datasets.data_source.base import DataSource, TrajectoryData

with initialize_config_dir(version_base=None, config_dir=str(REPO / 'src' / 'configs')):
    cfg = compose(
        config_name='config',
        overrides=['experiment=csgo', 'dataset=game/csgo', 'model=nanowm_s2_csgo'],
    )
assert cfg.dataset.name == 'csgo'
assert cfg.model.name == 'NanoWM-S-2'
config_check = subprocess.run(
    [sys.executable, 'src/main.py', '--cfg', 'job', 'experiment=csgo', 'dataset=game/csgo', 'model=nanowm_s2_csgo'],
    cwd=REPO, text=True, capture_output=True, check=True,
)
assert 'dataset:' in config_check.stdout and 'model:' in config_check.stdout

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
upstream_sha = '2ee3c355a6e44d2254eba8a8c29f199adddf4a3e'
report = {
    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'platform': platform.platform(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torchvision': metadata.version('torchvision'),
    'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': gpu_name,
    'vizdoom': metadata.version('vizdoom'),
    'hydra_core': metadata.version('hydra-core'),
    'diffusers': installed_versions['diffusers'],
    'verified_packages': installed_versions,
    'nanowm_revision': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(),
    'upstream_revision': upstream_sha,
    'config_smoke': 'passed',
}
print(json.dumps(report, indent=2, sort_keys=True))

In [ ]:
evidence_dir = REPO / 'reports' / 'evidence' / 'day1'
evidence_dir.mkdir(parents=True, exist_ok=True)
video_path = evidence_dir / 'pilot_episode_00000.mp4'
command = [
    sys.executable, 'src/scripts/collect_vizdoom.py',
    '--output-dir', str(PILOT_DIR),
    '--episodes', '10',
    '--base-seed', '42',
    '--policy', 'uniform',
    '--frame-skip', '4',
    '--resolution', '160x120',
    '--video-out', str(video_path),
]
completed = subprocess.run(command, cwd=REPO, text=True, capture_output=True, check=True)
print(completed.stdout)
summary = json.loads(completed.stdout.strip().splitlines()[-1])
assert summary['status'] == 'ok'
assert summary['episodes'] == 10
report['collector'] = summary
report['pilot_manifest'] = json.loads((PILOT_DIR / 'manifest.json').read_text())

In [ ]:
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(evidence_dir / 'pip-freeze.txt').write_text(freeze)
(evidence_dir / 'smoke-report.json').write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
(evidence_dir / 'pilot-manifest.json').write_text(
    json.dumps(report['pilot_manifest'], indent=2, sort_keys=True) + '\n'
)
assert video_path.exists() and 0 < video_path.stat().st_size < 5_000_000
assert len(report['pilot_manifest']['files']) == 10
print('Day 1 evidence written to', evidence_dir)
!git status --short